In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.default.bronze_credit")

# D2: cap utilization at 10 (keep over-limit signal, drop absurd values)
silver = silver.withColumn("RevolvingUtilizationOfUnsecuredLines",
    F.least(F.col("RevolvingUtilizationOfUnsecuredLines"), F.lit(10.0)))

# D3: drop impossible ages (n=1)
silver = silver.filter(F.col("age") >= 18)

In [0]:
# Spark has no simple .median() on a column — approxQuantile is the idiom
# (exact medians need a full sort across the cluster; approximate is standard at scale)
income_median = silver.approxQuantile("MonthlyIncome", [0.5], 0.01)[0]
print("median income:", income_median)

silver = (silver
    .withColumn("MonthlyIncome_missing", F.col("MonthlyIncome").isNull().cast("int"))
    .withColumn("MonthlyIncome", F.coalesce(F.col("MonthlyIncome"), F.lit(income_median)))
    .withColumn("NumberOfDependents", F.coalesce(F.col("NumberOfDependents"), F.lit(0))))

median income: 5400.0


In [0]:
# D7: collapse 96/98 codes (269 rows, per bronze DQ report) into "many times late"
for c in ["NumberOfTime30_59DaysPastDueNotWorse", "NumberOfTimes90DaysLate",
          "NumberOfTime60_89DaysPastDueNotWorse"]:
    silver = silver.withColumn(c, F.least(F.col(c), F.lit(20)))

silver = silver.withColumn("_silver_processed_at", F.current_timestamp())
silver.write.mode("overwrite").saveAsTable("workspace.default.silver_credit")
print("silver_credit written")

silver_credit written


In [0]:
s = spark.table("workspace.default.silver_credit")
print("rows:", s.count())                                                # expect 149,999
print("nulls remaining:", sum(s.filter(F.col(c).isNull()).count()
      for c in ["MonthlyIncome", "NumberOfDependents"]))                 # expect 0
print("max past-due:", s.agg(F.max("NumberOfTimes90DaysLate")).first()[0])  # expect 20
print("max utilization:", round(s.agg(F.max("RevolvingUtilizationOfUnsecuredLines")).first()[0], 2))  # expect 10.0

rows: 149999
nulls remaining: 0
max past-due: 20
max utilization: 10.0
